# 02. 결과 분석

모델 비교와 선정 근거, 피크 예측 성능, 오류 분석, 영향 변수를 정리한다. `python run.py`의 결과(`outputs/`)를 사용한다.

피크 지표는 일자별 상위 K개(기본 10개) 15분 구간으로 정의한다.
- `top10_hit`: 예측 상위 10개 구간 중 실제 상위 10개에 포함된 비율(순위 무관, 실제값 동률은 모두 포함)
- `top10_mae`: 실제·예측 상위 10개 값을 각각 내림차순 정렬해 순위끼리 비교한 MAE(발생 시각과 무관한 피크 수준 오차)

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.experiment import wavg, error_slices
from src.metrics import K, HIT_COL, TOPK_MAE_COL, daily_matrix, topk_scores
from src.report import cv_summary, lookback_table
from src.viz import setup_plot_style, heatmap, DOW_KR

setup_plot_style()
pd.set_option("display.max_columns", 50)
DATA, OUT = ROOT / C.DATA_PATH, ROOT / C.OUTPUT_DIR

required = ["metrics_cv.csv", "metrics_test.csv", "predictions_test.csv", "run_info.json"]
missing = [f for f in required if not (OUT / f).exists()]
if missing:
    raise FileNotFoundError(f"run.py 실행 결과가 없습니다: {missing}")

cv = pd.read_csv(OUT / "metrics_cv.csv")
test = pd.read_csv(OUT / "metrics_test.csv")
preds = pd.read_csv(OUT / "predictions_test.csv", parse_dates=["ts", "date"])
info = json.loads((OUT / "run_info.json").read_text(encoding="utf-8"))
selected = info["selected_model"]
print("GBM 백엔드:", info["gbm_backend"], "/ 선정 모델:", selected)

## 1. 교차검증

7/5부터 주 단위 walk-forward 7개 폴드. 폴드별 지표를 채점일 수로 가중평균한다. 선정 기준은 점예측 모델 중 CV MAE 최소이다.

In [ ]:
cv_mean = cv_summary(cv)
display(cv_mean[["model", "kind", "lookback", "mae", "rmse", "daily_max_mae", HIT_COL, TOPK_MAE_COL,
                 "mae_std_across_folds"]].round(3))

ax = cv.pivot(index="fold", columns="model", values="mae").plot(marker="o", figsize=(10, 4))
ax.set_ylabel("MAE (kW)")
ax.set_xlabel("검증 주 시작일")
ax.legend(fontsize=8)
plt.show()

08-02, 08-09 폴드는 휴가 주간과 그다음 주로, 7일 전 값에 의존하는 모델의 오차가 크게 증가한다.

## 2. 입력 범위 비교 (1d / 7d)

학습 데이터는 모두 예측일 이전 전체이며, 예측 시 입력으로 사용하는 과거 범위만 다르다.

In [ ]:
lb = lookback_table(cv_mean, test)
display(lb.round(3))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, col, title in [(axes[0], "cv_mae", "CV MAE"), (axes[1], f"test_{HIT_COL}", f"테스트 {HIT_COL}")]:
    lb.pivot(index="family", columns="lookback", values=col).plot(kind="bar", ax=ax, rot=0)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 3. 최종 테스트 (8/23~9/14)

In [ ]:
display(test[["model", "kind", "lookback", "mae", "rmse", "nmae", "bias", "daily_max_mae", "daily_max_bias",
              HIT_COL, TOPK_MAE_COL]].round(3))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
t = test.set_index("model")
for ax, col, xlabel in [(axes[0], "mae", "MAE (kW)"), (axes[1], HIT_COL, HIT_COL), (axes[2], TOPK_MAE_COL, f"{TOPK_MAE_COL} (kW)")]:
    t[col].sort_values().plot(kind="barh", ax=ax, color="#8aa")
    ax.set_xlabel(xlabel)
plt.tight_layout()
plt.show()

테스트 구간은 정상 운영 주간으로만 구성되어 휴가 주간이 포함된 CV보다 오차가 작다.

## 4. Ablation, 분할 단위 비교

In [ ]:
for name, title in [("ablation_cv.csv", "설정별 비교 (GBM, CV)"), ("leakage_demo.csv", "무작위 분할 단위별 MAE (GBM)")]:
    path = OUT / name
    if path.exists():
        print(title)
        display(pd.read_csv(path).round(3))

## 5. 테스트 예측

In [ ]:
w = preds[preds["ts"] < preds["ts"].min().normalize() + pd.Timedelta(days=14)].reset_index(drop=True)
a = daily_matrix(w, "kw").to_numpy()
p = daily_matrix(w, selected).to_numpy()
pos = lambda m: (np.arange(len(m))[:, None] * 96 + np.argsort(-m, axis=1, kind="stable")[:, :K]).ravel()
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(w["ts"], w["kw"], color="black", lw=1, label="실측")
ax.plot(w["ts"], w[selected], lw=1, label=selected)
ax.scatter(w.loc[pos(a), "ts"], w.loc[pos(a), "kw"], s=10, color="black", label=f"실제 상위 {K}")
ax.scatter(w.loc[pos(p), "ts"], w.loc[pos(p), selected], s=14, marker="x", color="tab:red", label=f"예측 상위 {K}")
ax.set_ylabel("15분 수요 (kW)")
ax.legend(ncol=4, fontsize=8, loc="upper right")
plt.show()

daily = preds.groupby("date")[["kw", selected]].max()
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [2, 1]})
axes[0].plot(daily.index, daily["kw"], "o-", color="black", ms=3, label="실측 일 최대")
axes[0].plot(daily.index, daily[selected], "o-", ms=3, label=f"{selected} 일 최대")
axes[0].set_ylabel("일 최대 15분 수요 (kW)")
axes[0].legend(fontsize=8)
axes[1].scatter(preds["kw"], preds[selected], s=3, alpha=0.3)
lim = [0, preds["kw"].max() * 1.05]
axes[1].plot(lim, lim, "r--", lw=1)
axes[1].set_xlabel("실측 (kW)")
axes[1].set_ylabel("예측 (kW)")
plt.tight_layout()
plt.show()

트리 모델은 잎에 속한 학습 타깃의 평균을 출력하므로 극단값이 완화되어 일 최대수요가 낮게 예측되는 경향(`daily_max_bias` 음수)이 있다.

## 6. 오류 분석

In [ ]:
sl = error_slices(preds, selected)
h = sl["by_hour"]
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(h["hour"], h["mae"], color="#8aa", label="MAE")
ax.plot(h["hour"], h["bias"], "o-", color="tab:red", label="bias")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("시간")
ax.set_ylabel("kW")
ax.legend(fontsize=8)
plt.show()

display(sl["by_dow"].assign(요일=lambda d: d["dow"].map(DOW_KR)).round(2))
display(sl["by_prod"].round(2))

### 피크 시간대 적중

실제 상위 10개 구간과 예측 상위 10개 구간의 시간 분포를 비교한다.

In [ ]:
A = daily_matrix(preds, "kw")
P = daily_matrix(preds, selected)
top_a = np.argsort(-A.to_numpy(), axis=1, kind="stable")[:, :K] // 4
top_p = np.argsort(-P.to_numpy(), axis=1, kind="stable")[:, :K] // 4
dist = pd.DataFrame({"실제": np.bincount(top_a.ravel(), minlength=24),
                     "예측": np.bincount(top_p.ravel(), minlength=24)})
ax = dist.plot(kind="bar", figsize=(11, 3.5), rot=0)
ax.set_xlabel("시간")
ax.set_ylabel("상위 구간 수")
plt.show()

bd = sl["by_day"].assign(요일=lambda d: d["dow"].map(DOW_KR))
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(pd.to_datetime(bd["date"]), bd[f"top{K}_hits"], color="#8aa")
ax.set_ylim(0, K)
ax.set_ylabel(f"상위 {K}개 적중 수")
plt.show()
display(bd.sort_values(f"top{K}_hits").head(8).round(2))

## 7. 영향 변수와 상호작용

최종 모델을 같은 설정으로 재학습하고 `run.py` 예측과의 일치를 확인한다. 순열 중요도와 PDP는 행 단위 트리 모델에 적용하며, 선정 모델이 시퀀스 모델이면 GBM으로 분석한다.

In [ ]:
from src.data import load_raw, clean
from src.features import to_long, build_features, feature_columns
from src.experiment import train_test, fit_predict_model
from src.models import make_models

hourly, _, _ = clean(load_raw(DATA))
feat = build_features(to_long(hourly))
tr, te = train_test(feat)
kind, model, lookback = make_models()[selected]
p, _ = fit_predict_model(model, lookback, feat, tr, te)
print(f"{selected}: run.py 예측과의 최대 차이 {float(np.abs(p - preds[selected].to_numpy()).max()):.2e}")

analysis_name = selected
if getattr(model, "needs_full_frame", False):
    analysis_name = "gbm"
    kind, model, lookback = make_models()[analysis_name]
    fit_predict_model(model, lookback, feat, tr, te)
cols = feature_columns(lookback=lookback)
print("분석 모델:", analysis_name, f"(입력 범위 {lookback}, 피처 {len(cols)}개)")

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

pi = permutation_importance(model, te[cols], te["kw"], scoring="neg_mean_absolute_error",
                            n_repeats=5, random_state=C.SEED)
imp = pd.Series(pi.importances_mean, index=cols).sort_values()
ax = imp.plot(kind="barh", figsize=(7, 6), color="#8aa")
ax.set_xlabel("순열 시 MAE 증가 (kW)")
plt.show()

In [ ]:
# 정수형 피처는 PDP 격자 생성 시 반올림 문제가 있어 실수형으로 변환한다(트리 예측에는 영향 없음).
X = te[cols].sample(min(1000, len(te)), random_state=C.SEED).astype(float)
one_d = [c for c in ["hour", "lag_7d", "lag_1d", "prev_day_max", "temp", "prod_h"] if c in cols][:3]
two_d = [pair for pair in [("hour", "dow"), ("hour", "prod_h"), ("hour", "prev_is_off_day"), ("hour", "lag_1d")]
         if all(c in cols for c in pair)][:2]

fig, axes = plt.subplots(1, len(one_d), figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(model, X, one_d, ax=axes)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(two_d), figsize=(13, 4.5))
PartialDependenceDisplay.from_estimator(model, X, two_d, ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample = te[cols].sample(min(2000, len(te)), random_state=C.SEED)
    shap.summary_plot(explainer.shap_values(sample), sample, show=False)
    plt.show()
except ImportError:
    print("shap 미설치")
except Exception as e:
    print("SHAP 생략:", e)

## 8. 피크 시간대 예측 캘린더

일자·시간별로 예측 상위 10개 구간 수(0~4)를 표시하고, 실제 상위 10개 구간이 있는 시간을 ×로 표시한다.

In [ ]:
cnt_p = np.zeros((len(P), 24))
cnt_a = np.zeros((len(A), 24))
np.add.at(cnt_p, (np.arange(len(P))[:, None].repeat(K, 1), top_p), 1)
np.add.at(cnt_a, (np.arange(len(A))[:, None].repeat(K, 1), top_a), 1)
labels = [f"{d:%m-%d} ({DOW_KR[d.dayofweek + 1]})" for d in P.index]
fig, ax = plt.subplots(figsize=(13, 8))
heatmap(pd.DataFrame(cnt_p, index=labels, columns=range(24)), ax=ax, cmap="Reds",
        cbar_label=f"예측 상위 {K}개 구간 수 ({selected})")
yy, xx = np.where(cnt_a > 0)
ax.scatter(xx, yy, marker="x", color="black", s=18, label=f"실제 상위 {K}개 포함 시간")
ax.set_xlabel("시간")
ax.legend(loc="upper right", fontsize=8)
plt.show()